In [0]:

# Constantes del laboratorio — usar en todos los notebooks de tareas
CATALOG       = "dbassociate"
SCHEMA_BRONZE = "bronze"
SCHEMA_SILVER = "silver"
SCHEMA_GOLD   = "gold"
VOLUME_PATH   = "/Volumes/dbassociate/default/vol_landing"
SOURCE_PATH   = f"{VOLUME_PATH}/sesion09"

BRONZE_TABLE  = f"{CATALOG}.{SCHEMA_BRONZE}.transacciones_retail_raw"
SILVER_TABLE  = f"{CATALOG}.{SCHEMA_SILVER}.transacciones_clean"
GOLD_TABLE    = f"{CATALOG}.{SCHEMA_GOLD}.ventas_por_tienda"

print(f"Source  : {SOURCE_PATH}")
print(f"Bronze  : {BRONZE_TABLE}")
print(f"Silver  : {SILVER_TABLE}")
print(f"Gold    : {GOLD_TABLE}")

In [0]:
from pyspark.sql.functions import col, upper, trim, when, round as spark_round

# Leer Task Value de bronze_load
# debugValue se usa cuando se ejecuta el notebook fuera del Workflow
bronze_count = dbutils.jobs.taskValues.get(
    taskKey="bronze_load",
    key="bronze_count",
    default=0,
    debugValue=35
)
print(f"Registros recibidos desde bronze: {bronze_count}")

df_bronze = spark.table(BRONZE_TABLE)

df_silver = (
    df_bronze
    # Reglas de calidad
    .filter(col("order_id").isNotNull())
    .filter(col("total") > 0)
    # Normalización de strings
    .withColumn("categoria",   upper(trim(col("categoria"))))
    .withColumn("estado",      upper(trim(col("estado"))))
    .withColumn("metodo_pago", upper(trim(col("metodo_pago"))))
    # Columnas derivadas
    .withColumn("tiene_descuento",
        when(col("descuento_pct") > 0, True).otherwise(False))
    .withColumn("monto_descuento",
        spark_round(
            col("precio_unit") * col("cantidad") * col("descuento_pct") / 100, 2))
    # Solo transacciones activas en Silver
    .filter(col("estado").isin("COMPLETADO", "PENDIENTE"))
    # Limpiar columnas técnicas de Bronze
    .drop("_source_file", "_batch_session")
    .withColumnRenamed("_ingestion_ts", "bronze_ts")
)

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

silver_count = spark.table(SILVER_TABLE).count()
print(f"Silver generado: {silver_count} registros validos -> {SILVER_TABLE}")
print(f"Registros excluidos (devueltos/invalidos): {bronze_count - silver_count}")

dbutils.jobs.taskValues.set(key="silver_count", value=silver_count)